# MetroPulse — MCNN training (Colab GPU)

Same code as `pipeline/mcnn/` in the project — this notebook just writes those files out here and runs `train.py` on a Colab GPU instead of the local CPU. Checkpoints save to Google Drive as it goes, so a Colab disconnect doesn't lose progress.

**Before running:** Runtime -> Change runtime type -> T4 GPU.


In [ ]:
import torch
print('GPU available:', torch.cuda.is_available())
print(torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'no GPU — set Runtime > Change runtime type > T4 GPU')


## 1. Dataset

Upload `ShanghaiTech.zip` (the same file you downloaded from Kaggle) to your Google Drive first (any folder — set the path below to match), **or** run the cell after this one to upload it directly from your computer instead (slower, no Drive needed).


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# EDIT this to wherever you put the zip in your Drive
DRIVE_ZIP_PATH = '/content/drive/MyDrive/ShanghaiTech.zip'
CHECKPOINT_DIR = '/content/drive/MyDrive/metropulse_checkpoints'


**Option A — from Drive** (run this cell, skip the upload cell below):

In [ ]:
import shutil, os
assert os.path.exists(DRIVE_ZIP_PATH), f'{DRIVE_ZIP_PATH} not found — check the path or use the upload cell instead'
shutil.unpack_archive(DRIVE_ZIP_PATH, '/content/')
print('extracted to /content/ShanghaiTech')


**Option B — direct upload** (skip if you used Option A):

In [ ]:
from google.colab import files
import shutil
uploaded = files.upload()  # pick ShanghaiTech.zip from your computer
shutil.unpack_archive(next(iter(uploaded)), '/content/')
print('extracted to /content/ShanghaiTech')


## 2. Pipeline code — identical to `pipeline/mcnn/` in the project

In [ ]:
%%writefile model.py
"""
Multi-Column CNN (Zhang et al., "Single-Image Crowd Counting via Multi-Column
Convolutional Neural Network", CVPR 2016). Three parallel columns with
different receptive field sizes (large/medium/small filters) capture heads at
different scales from perspective distortion, fused into a single-channel
density map. Two stride-2 maxpools per column mean the output is 1/4 the
input resolution in each dimension — the ground truth density maps built by
prepare_density_maps.py are downsampled to match.
"""

import torch
import torch.nn as nn


class _Column(nn.Module):
    def __init__(self, c1, c2, c3, c4, k1, k2, k3, k4):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(3, c1, k1, padding=k1 // 2), nn.ReLU(inplace=True),
            nn.MaxPool2d(2),
            nn.Conv2d(c1, c2, k2, padding=k2 // 2), nn.ReLU(inplace=True),
            nn.MaxPool2d(2),
            nn.Conv2d(c2, c3, k3, padding=k3 // 2), nn.ReLU(inplace=True),
            nn.Conv2d(c3, c4, k4, padding=k4 // 2), nn.ReLU(inplace=True),
        )

    def forward(self, x):
        return self.net(x)


class MCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.large = _Column(16, 32, 16, 8, 9, 7, 7, 7)
        self.medium = _Column(20, 40, 20, 10, 7, 5, 5, 5)
        self.small = _Column(24, 48, 24, 12, 5, 3, 3, 3)
        self.fuse = nn.Conv2d(8 + 10 + 12, 1, kernel_size=1)

    def forward(self, x):
        feats = torch.cat([self.large(x), self.medium(x), self.small(x)], dim=1)
        return self.fuse(feats)


In [ ]:
%%writefile density_map.py
"""
Ground-truth density map generation from head-point annotations. Part B is
street scenes with sparse, relatively uniform crowds (unlike Part A's dense,
perspective-heavy scenes), so the original MCNN paper uses one fixed-sigma
Gaussian per head rather than Part A's geometry-adaptive (kNN-distance)
kernel — sigma=15 is the value used across common reference implementations
for Part B.
"""

import numpy as np
from scipy.ndimage import gaussian_filter

SIGMA = 15.0


def points_to_density_map(points, shape, sigma=SIGMA):
    """points: (N, 2) array of (x, y) head locations. shape: (h, w).
    Summing a delta per point then blurring once is equivalent to summing N
    individually-blurred Gaussians, and is far cheaper than N separate
    convolutions over the full image."""
    density = np.zeros(shape, dtype=np.float32)
    h, w = shape
    if points is None or len(points) == 0:
        return density
    for x, y in points:
        xi = min(w - 1, max(0, int(round(float(x)))))
        yi = min(h - 1, max(0, int(round(float(y)))))
        density[yi, xi] += 1.0
    # mode='constant' truncates the kernel at the image border, so a small
    # amount of mass from heads near the edge is lost rather than reflected
    # back in — matches standard practice in MCNN reference implementations.
    return gaussian_filter(density, sigma=sigma, mode="constant")


In [ ]:
%%writefile dataset.py
"""
Loads a ShanghaiTech Part B split. Expects density_maps/ to already exist
(run prepare_density_maps.py first) — this class doesn't generate them
itself, so a missing cache fails loudly instead of silently regenerating
slow ground truth mid-epoch.
"""

from pathlib import Path

import numpy as np
import torch
from PIL import Image
from torch.utils.data import Dataset


class ShanghaiTechB(Dataset):
    def __init__(self, root, split):
        root = Path(root)
        self.img_dir = root / f"{split}_data" / "images"
        self.den_dir = root / f"{split}_data" / "density_maps"
        self.paths = sorted(self.img_dir.glob("*.jpg"))
        if not self.paths:
            raise FileNotFoundError(f"no images found in {self.img_dir}")

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, idx):
        img_path = self.paths[idx]
        den_path = self.den_dir / f"{img_path.stem}.npy"
        if not den_path.exists():
            raise FileNotFoundError(f"{den_path} missing — run prepare_density_maps.py first")

        img = Image.open(img_path).convert("RGB")
        img_t = torch.from_numpy(np.asarray(img, dtype=np.float32) / 255.0).permute(2, 0, 1)

        density = np.load(den_path)
        # crop to a multiple of 4 so the image and the model's 1/4-resolution
        # output line up, then downsample the density map by summing 4x4
        # blocks — summing (not averaging) preserves total head count
        h, w = density.shape
        h4, w4 = h - h % 4, w - w % 4
        density = density[:h4, :w4].reshape(h4 // 4, 4, w4 // 4, 4).sum(axis=(1, 3))
        den_t = torch.from_numpy(density).unsqueeze(0)
        img_t = img_t[:, :h4, :w4]

        return img_t, den_t


In [ ]:
%%writefile prepare_density_maps.py
"""
Precomputes and caches ground-truth density maps as .npy files, once, so
training doesn't redo the scipy Gaussian filtering on every epoch. Run this
before train.py.
"""

import argparse
from pathlib import Path

import numpy as np
import scipy.io as sio
from PIL import Image

from density_map import points_to_density_map


def process_split(root: Path, split: str) -> None:
    img_dir = root / f"{split}_data" / "images"
    gt_dir = root / f"{split}_data" / "ground-truth"
    out_dir = root / f"{split}_data" / "density_maps"
    out_dir.mkdir(exist_ok=True)

    img_paths = sorted(img_dir.glob("*.jpg"))
    if not img_paths:
        raise FileNotFoundError(f"no images found in {img_dir}")

    made = 0
    for img_path in img_paths:
        out_path = out_dir / f"{img_path.stem}.npy"
        if out_path.exists():
            continue
        gt_path = gt_dir / f"GT_{img_path.stem}.mat"
        with Image.open(img_path) as img:
            w, h = img.size
        mat = sio.loadmat(gt_path)
        points = mat["image_info"][0, 0][0, 0][0]
        density = points_to_density_map(points, (h, w))
        np.save(out_path, density)
        made += 1

    print(f"{split}: {len(img_paths)} images, {made} density maps written -> {out_dir}")


def main() -> None:
    ap = argparse.ArgumentParser()
    ap.add_argument("--data-root", default="../data/ShanghaiTech/part_B")
    args = ap.parse_args()
    root = Path(args.data_root)

    process_split(root, "train")
    process_split(root, "test")


if __name__ == "__main__":
    main()


In [ ]:
%%writefile train.py
"""
MCNN training on ShanghaiTech Part B. Runs on CPU or GPU automatically
(same script for a local CPU run or a Colab GPU session). Batch size 1 —
images vary in size, and padding/cropping to a fixed batch shape isn't worth
the complexity for a Phase-I proof of concept on 400 training images.
Run prepare_density_maps.py first.
"""

import argparse
import csv
import time
from pathlib import Path

import torch
import torch.nn as nn
from torch.utils.data import DataLoader

from dataset import ShanghaiTechB
from model import MCNN


def evaluate(model, loader, device):
    model.eval()
    mae, mse, n = 0.0, 0.0, 0
    with torch.no_grad():
        for img, den in loader:
            img, den = img.to(device), den.to(device)
            pred = model(img)
            pred_count = pred.sum().item()
            gt_count = den.sum().item()
            mae += abs(pred_count - gt_count)
            mse += (pred_count - gt_count) ** 2
            n += 1
    model.train()
    return mae / n, (mse / n) ** 0.5


def main() -> None:
    ap = argparse.ArgumentParser()
    ap.add_argument("--data-root", default="../data/ShanghaiTech/part_B")
    ap.add_argument("--epochs", type=int, default=100)
    ap.add_argument("--lr", type=float, default=1e-5)
    ap.add_argument("--eval-every", type=int, default=5)
    ap.add_argument("--out-dir", default="./checkpoints")
    ap.add_argument("--resume", action="store_true",
                     help="continue from checkpoints/resume.pt if it exists")
    args = ap.parse_args()

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"device: {device}", flush=True)
    train_set = ShanghaiTechB(args.data_root, "train")
    test_set = ShanghaiTechB(args.data_root, "test")
    train_loader = DataLoader(train_set, batch_size=1, shuffle=True)
    test_loader = DataLoader(test_set, batch_size=1, shuffle=False)

    model = MCNN().to(device)
    opt = torch.optim.Adam(model.parameters(), lr=args.lr)
    loss_fn = nn.MSELoss()

    out_dir = Path(args.out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)
    log_path = out_dir / "metrics.csv"
    resume_path = out_dir / "resume.pt"
    best_mae = float("inf")
    start_epoch = 1

    # a laptop can sleep, lose power, or just get closed mid-run — checkpoint
    # every epoch (not just on eval improvement) so at most one epoch of
    # progress is ever at risk, and --resume can pick back up from it
    if args.resume and resume_path.exists():
        ckpt = torch.load(resume_path, map_location=device)
        model.load_state_dict(ckpt["model"])
        opt.load_state_dict(ckpt["optimizer"])
        best_mae = ckpt["best_mae"]
        start_epoch = ckpt["epoch"] + 1
        print(f"resuming from epoch {start_epoch} (best_mae so far: {best_mae:.2f})", flush=True)
    else:
        with open(log_path, "w", newline="") as f:
            csv.writer(f).writerow(["epoch", "train_loss", "mae", "rmse", "seconds"])

    for epoch in range(start_epoch, args.epochs + 1):
        t0 = time.time()
        total_loss = 0.0
        for img, den in train_loader:
            img, den = img.to(device), den.to(device)
            opt.zero_grad()
            pred = model(img)
            loss = loss_fn(pred, den)
            loss.backward()
            opt.step()
            total_loss += loss.item()
        avg_loss = total_loss / len(train_loader)
        elapsed = time.time() - t0

        mae = rmse = float("nan")
        if epoch % args.eval_every == 0 or epoch == args.epochs:
            mae, rmse = evaluate(model, test_loader, device)
            if mae < best_mae:
                best_mae = mae
                torch.save(model.state_dict(), out_dir / "best.pt")

        with open(log_path, "a", newline="") as f:
            csv.writer(f).writerow([epoch, f"{avg_loss:.4f}", f"{mae:.2f}", f"{rmse:.2f}", f"{elapsed:.1f}"])
        print(f"epoch {epoch:3d}/{args.epochs}  loss={avg_loss:.4f}  mae={mae:.2f}  rmse={rmse:.2f}  {elapsed:.1f}s", flush=True)

        torch.save({
            "epoch": epoch, "model": model.state_dict(),
            "optimizer": opt.state_dict(), "best_mae": best_mae,
        }, resume_path)

    torch.save(model.state_dict(), out_dir / "last.pt")


if __name__ == "__main__":
    main()


## 3. Precompute ground-truth density maps (one-time, ~1-2 min on Colab)

In [ ]:
!python prepare_density_maps.py --data-root /content/ShanghaiTech/part_B


## 4. Train

Checkpoints (`best.pt`, `last.pt`, `resume.pt`, `metrics.csv`) go straight to Drive, so if Colab disconnects you, re-run this cell — `--resume` picks up from the last completed epoch.


In [ ]:
!python train.py --data-root /content/ShanghaiTech/part_B --epochs 100 --eval-every 5 --out-dir "$CHECKPOINT_DIR" --resume


## 5. Bring it home

`best.pt` is already in your Drive at `CHECKPOINT_DIR` — sync Drive to this PC (or download it manually from drive.google.com) and drop it into `pipeline/mcnn/checkpoints/best.pt` in the project. Tell Claude once it's there and it'll pick up from it.
